<a href="https://colab.research.google.com/github/nvaprameya-source/AI-Travel-Concierge-AI-Agent-Development-Dual-Track-Version/blob/main/framework_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai gradio faiss-cpu sentence-transformers numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.6 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
GROQ_API_KEY=userdata.get('GROQ_API_KEY')

In [ ]:
import json

frameworks = [
    {"type": "itinerary", "content": "For a 3-day trip: Day 1 arrival and light exploration, Day 2 full exploration, Day 3 shopping and departure"},
    {"type": "itinerary", "content": "For short trips: prioritize nearby attractions and minimize travel time"},
    {"type": "budget", "content": "Split budget: 40% stay, 30% food, 20% transport, 10% activities"},
    {"type": "budget", "content": "For low budget travel: use hostels, public transport, and free attractions"},
    {"type": "group_travel", "content": "For group trips: use shared stays and cost splitting"},
    {"type": "solo_travel", "content": "For solo trips: prioritize safety and flexibility"},
    {"type": "activity_mapping", "content": "Adventure: trekking, camping. Relaxation: cafes, sightseeing. Cultural: temples, markets"},
    {"type": "season", "content": "In summer: prefer morning/evening activities, avoid midday heat"}
]

with open("frameworks.json", "w") as f:
    json.dump(frameworks, f)

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Load frameworks
import json
with open("frameworks.json", "r") as f:
    data = json.load(f)

texts = [item["content"] for item in data]

# Create embeddings
embeddings = model.encode(texts)

# Create FAISS index
dimension = len(embeddings[0])
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

def retrieve_frameworks(query, k=3):
    query_embedding = model.encode([query])
    distances, indices = index.search(np.array(query_embedding), k)
    return "\n".join([texts[i] for i in indices[0]])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
!pip install langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.2 MB/s eta 0:00:00


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=GROQ_API_KEY,
    temperature=0.3,
    max_tokens=500
)

In [ ]:
conversation_history = []

SYSTEM_PROMPT = """
You are an AI travel planner.

Use the provided frameworks to generate structured, practical, and realistic travel plans.

Rules:
- Always give a day-wise itinerary
- Include budget breakdown
- Include useful tips
- Avoid hallucinating unknown place names
"""


def agent_loop(user_input):

    # 🔍 Step 1: Retrieve frameworks (RAG)
    context = retrieve_frameworks(user_input)

    # 🧠 Step 2: Build messages
    messages = []

    messages.append({
        "role": "system",
        "content": SYSTEM_PROMPT
    })

    messages.append({
        "role": "system",
        "content": f"Relevant frameworks:\n{context}"
    })

    messages.extend(conversation_history)

    messages.append({
        "role": "user",
        "content": user_input
    })

    # 🔁 Convert messages → prompt (Groq format)
    prompt = ""
    for msg in messages:
        prompt += f"{msg['role'].upper()}: {msg['content']}\n"

    # 🧠 Step 3: Single LLM call
    response = llm.invoke(prompt)

    reply = response.content

    # 💾 Step 4: Save memory
    conversation_history.append({
        "role": "user",
        "content": user_input
    })

    conversation_history.append({
        "role": "assistant",
        "content": reply
    })

    return reply

In [ ]:
print(agent_loop("Plan a 3-day trip to Goa under 8000 with friends"))

**Goa 3-Day Trip Plan**

**Day 1: Arrival and Light Exploration**

* Morning: Arrive at Dabolim Airport (GOI) or Madgaon Railway Station
* 10:00 AM: Check-in at a budget-friendly hostel in Panjim ( approx. ₹800 per night)
* 11:30 AM: Visit the **Miramar Beach** (free entry) for a relaxing morning stroll
* 1:00 PM: Have lunch at a local eatery in Panjim (approx. ₹200 per meal)
* 2:30 PM: Explore the **Panjim Market** (free entry) for some shopping and local experiences
* 5:00 PM: Visit the **Church of St. Francis of Assisi** (free entry) for a peaceful evening
* 7:00 PM: Enjoy a seafood dinner at a local restaurant (approx. ₹500 per meal)

**Day 2: Full Exploration**

* 8:00 AM: Start the day with a delicious breakfast at a local café (approx. ₹150 per meal)
* 9:30 AM: Visit the **Basilica of Bom Jesus** (entry fee: ₹10) in Old Goa
* 11:30 AM: Explore the **Se Cathedral** (entry fee: ₹10) in Old Goa
* 1:00 PM: Have lunch at a local eatery in Old Goa (approx. ₹200 per meal)
* 2:30 PM: Vi